In [1]:
from pathlib import Path
import pandas as pd
import altair as alt

In [3]:
print("Carpeta actual:", Path.cwd())
folder = Path("../data/enoe_2026_trim2_csv")
print("Existe la carpeta:", folder.exists())
print("Archivos CSV:", [p.name for p in folder.rglob("*") if p.suffix.lower() == ".csv"])

Carpeta actual: /Users/jocelynsanchez/Q4/Data_viz/project/data-viz-project/notebooks
Existe la carpeta: True
Archivos CSV: ['ENOE_COE2T226.csv', 'ENOE_COE1T226.csv', 'ENOE_HOGT226.csv', 'ENOE_SDEMT226.csv', 'ENOE_VIVT226.csv']


In [5]:
file = folder / "ENOE_SDEMT226.csv"
enoe = pd.read_csv(file, encoding="latin-1", low_memory=False)
enoe.columns = enoe.columns.str.strip().str.lower()

print("Dimensiones:", enoe.shape)
print(enoe.columns.tolist())
enoe.head()
print("\n".join(enoe.columns))

Dimensiones: (417128, 115)
['r_def', 'cve_loc', 'cve_mun', 'est', 'est_d_tri', 'est_d_men', 'cve_ageb', 't_loc_tri', 't_loc_men', 'cd_a', 'cve_ent', 'con', 'upm', 'd_sem', 'n_pro_viv', 'v_sel', 'n_hog', 'h_mud', 'n_ent', 'per', 'n_ren', 'c_res', 'par_c', 'sex', 'eda', 'nac_dia', 'nac_mes', 'nac_anio', 'l_nac_c', 'cs_p12', 'cs_p13_1', 'cs_p13_2', 'cs_p14_c', 'cs_p15', 'cs_p16', 'cs_p17', 'n_hij', 'e_con', 'cs_p20a_1', 'cs_p20a_c', 'cs_p20b_1', 'cs_p20b_c', 'cs_p20c_1', 'cs_ad_mot', 'cs_p21_des', 'cs_ad_des', 'cs_nr_mot', 'cs_p23_des', 'cs_nr_ori', 'ur', 'zona', 'salario', 'fac_tri', 'fac_men', 'clase1', 'clase2', 'clase3', 'pos_ocu', 'seg_soc', 'rama', 'c_ocu11c', 'ing7c', 'dur9c', 'emple7c', 'medica5c', 'buscar5c', 'rama_est1', 'rama_est2', 'dur_est', 'ambito1', 'ambito2', 'tue1', 'tue2', 'tue3', 'busqueda', 'd_ant_lab', 'd_cexp_est', 'dur_des', 'sub_o', 's_clasifi', 'remune2c', 'pre_asa', 'tip_con', 'dispo', 'nodispo', 'c_inac5c', 'pnea_est', 'niv_ins', 'eda5c', 'eda7c', 'eda12c', 'ed

In [6]:
# Inspect codes before filtering or assigning labels.
for col in ["r_def", "c_res", "clase2", "sex", "niv_ins", "ing7c"]:
    print(f"\n{col}:")
    print(enoe[col].value_counts(dropna=False).sort_index().to_string())

print("\nIngreso mensual:")
print(enoe["ingocup"].describe().to_string())


r_def:
r_def
0     416393
15       735

c_res:
c_res
1    400746
2      9305
3      7077

clase2:
clase2
0     72419
1    190325
2      5217
3     15408
4    133759

sex:
sex
       9305
1    195526
2    212297

niv_ins:
niv_ins
0     30436
1     77040
2     64683
3    106840
4    137270
5       859

ing7c:
ing7c
0    226803
1     85193
2     61329
3     11241
4      4349
5      1236
6      7868
7     19109

Ingreso mensual:
count    417128.000000
mean       3372.287415
std        7144.343731
min           0.000000
25%           0.000000
50%           0.000000
75%        4300.000000
max      250000.000000


In [7]:
cols = ["r_def", "c_res", "eda", "clase2", "sex", "niv_ins",
        "cs_p13_1", "cs_p13_2", "cs_p15", "ing7c", "ingocup", "fac_tri"]
enoe[cols] = enoe[cols].apply(pd.to_numeric, errors="coerce")

# Completed interviews, current residents, occupied population aged 15+.
workers = enoe.loc[
    enoe["r_def"].eq(0) & enoe["c_res"].isin([1, 3])
    & enoe["eda"].between(15, 98) & enoe["clase2"].eq(1)
].copy()

print("Personas ocupadas en la muestra:", len(workers))
print("\nNivel escolar y años aprobados:")
print(pd.crosstab(workers["cs_p13_1"], workers["cs_p13_2"]).to_string())
print("\nIngresos por clasificación:")
print(workers.groupby("ing7c")["ingocup"].agg(["count", "min", "max"]).to_string())

Personas ocupadas en la muestra: 189353

Nivel escolar y años aprobados:
cs_p13_2   0.0   1.0   2.0    3.0    4.0    5.0    6.0  7.0  8.0  9.0
cs_p13_1                                                             
0.0       4015     0     0      0      0      0      0    0    0    0
1.0          0    20    31     46      0      0      0    0    0    1
2.0          0   763  1732   2787   1396   1297  19579    0    0   14
3.0          0  1248  2659  46946      0      0      0    0    0    5
4.0          0  3080  5361  38991      0      0      0    0    0    5
5.0          0     3     8     58    140      0      0    0    0    0
6.0          0   204  1139   3755     20      2      0    0    0    1
7.0          0  2122  2752   3617  19493  19982    570   53    2   76
8.0          0   262  3089    735     19      4      1    5    0    9
9.0          0    50   409    270     34      7      2    0    0    2
99.0         0     0     0      0      0      0      0    0    0  482

Ingresos por cla

In [9]:
education_labels = {
    0: "Primary or less", 1: "Primary or less", 2: "Primary or less",
    3: "Secondary", 4: "Upper secondary",
    5: "Technical / teacher training", 6: "Technical / teacher training",
    7: "University", 8: "Postgraduate", 9: "Postgraduate"
}

workers["education"] = workers["cs_p13_1"].map(education_labels)
workers["gender"] = workers["sex"].map({1: "Men", 2: "Women"})

# Exclude unknown schooling years, unknown sex and invalid weights.
eligible = workers.loc[
    workers["education"].notna() & workers["gender"].notna()
    & workers["cs_p13_2"].between(0, 8) & workers["fac_tri"].gt(0)
].copy()

# Keep workers with valid weekly hours of at least 35.
eligible = eligible.loc[
    pd.to_numeric(eligible["hrsocup"], errors="coerce").between(35, 168)
].copy()

# Average monthly income among workers reporting a positive amount.
reported = eligible.loc[eligible["ingocup"].gt(0)].copy()
reported["weighted_income"] = reported["ingocup"] * reported["fac_tri"]

income = (
    reported.groupby(["education", "gender"])
    .agg(weighted_income=("weighted_income", "sum"),
         weight=("fac_tri", "sum"), sample_n=("ingocup", "size"))
    .reset_index()
)
income["monthly_income"] = income["weighted_income"] / income["weight"]

print(income[["education", "gender", "monthly_income", "sample_n"]]
      .round(1).to_string(index=False))

                   education gender  monthly_income  sample_n
                Postgraduate    Men         30237.2       871
                Postgraduate  Women         24291.9       836
             Primary or less    Men          9952.7      9995
             Primary or less  Women          7867.0      4382
                   Secondary    Men         11631.4     17413
                   Secondary  Women          9232.6      8589
Technical / teacher training    Men         14548.9      1033
Technical / teacher training  Women         11468.7      1208
                  University    Men         18312.7     10740
                  University  Women         15283.3      8427
             Upper secondary    Men         12552.8     15189
             Upper secondary  Women         10260.6      8195


In [10]:
coverage = eligible.groupby(["education", "gender"])["fac_tri"].sum()
response = reported.groupby(["education", "gender"])["fac_tri"].sum()

print((100 * response / coverage).round(1)
      .rename("positive_income_reported_pct").to_string())

education                     gender
Postgraduate                  Men       48.4
                              Women     49.8
Primary or less               Men       71.1
                              Women     74.3
Secondary                     Men       70.6
                              Women     72.2
Technical / teacher training  Men       60.4
                              Women     57.8
University                    Men       54.3
                              Women     52.9
Upper secondary               Men       64.7
                              Women     65.5


In [11]:
education_order = [
    "Primary or less", "Secondary", "Upper secondary",
    "Technical / teacher training", "University", "Postgraduate"
]
colors = alt.Scale(domain=["Women", "Men"], range=["#8624F5", "#1FC3AA"])

# Relative income gap, using men's income as the reference.
income_gaps = income.pivot(index="education", columns="gender",
                          values="monthly_income").reset_index()
income_gaps["gap"] = (income_gaps["Men"] - income_gaps["Women"]) / income_gaps["Men"]
income_gaps["gap_label"] = income_gaps["gap"].map(lambda x: f"{x:.1%}")

base = alt.Chart(income).encode(
    y=alt.Y("education:N", sort=education_order, title=None,
            axis=alt.Axis(labelLimit=240, labelPadding=10)),
    x=alt.X("monthly_income:Q", title="Average monthly labor income (MXN)",
            scale=alt.Scale(domain=[0, 35000]),
            axis=alt.Axis(values=list(range(0, 35001, 5000)), format="$,.0f"))
)

lines = base.mark_line(color="#B0B0B0", strokeWidth=3).encode(
    detail="education:N", order="monthly_income:Q"
)

dots = base.mark_circle(size=170, opacity=1).encode(
    color=alt.Color("gender:N", scale=colors,
                    legend=alt.Legend(title=None, orient="bottom",
                                      direction="horizontal"))
)

women_labels = (
    base.transform_filter(alt.datum.gender == "Women")
    .mark_text(dy=-17, fontSize=12, color="#8624F5")
    .encode(text=alt.Text("monthly_income:Q", format="$,.0f"))
)

men_labels = (
    base.transform_filter(alt.datum.gender == "Men")
    .mark_text(dy=18, fontSize=12, color="#1FC3AA")
    .encode(text=alt.Text("monthly_income:Q", format="$,.0f"))
)

gaps = (
    alt.Chart(income_gaps).mark_text(fontSize=13, fontWeight="bold",
                                     color="#C0392B")
    .encode(
        y=alt.Y("education:N", sort=education_order, title=None, axis=None),
        text="gap_label:N"
    )
    .properties(width=95, height=360, title=alt.TitleParams(
        text=["Income gap (%)", "(Men − Women) / Men"],
        fontSize=11, color="#555555"
    ))
)

income_chart = (
    alt.hconcat(
        (lines + dots + women_labels + men_labels).properties(width=550, height=360),
        gaps, spacing=8
    )
    .resolve_scale(y="shared")
    .properties(title=alt.TitleParams(
        text="Gender gaps in monthly income by education",
        subtitle=["Mexico, Q2 2026 | Workers aged 15+ working 35+ hours per week",
                  "Average among those reporting positive monthly labor income"],
        anchor="start", fontSize=22, fontWeight="bold",
        subtitleFontSize=13, subtitleColor="#666666",
        subtitlePadding=8, offset=20
    ))
    .configure_axis(labelFontSize=12, titleFontSize=13, titlePadding=12, grid=False)
    .configure_legend(labelFontSize=12, symbolSize=100)
    .configure_view(stroke=None)
)

income_chart

alt.HConcatChart(...)